<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">11 Lab 2 - Cost</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Lab - Cost
#### Duration: 20 minutes

**In a live class:** You can complete this lab after class.

## Overview

The monthly Databricks bill for CloudField jumped last month. No new workloads shipped and no new customers onboarded, so the increase has no obvious explanation.

Your job is to find what is eating the budget and fix it. This is a lab, so you diagnose and fix it yourself.

You will:

- Query the cost snapshot to rank every workload by dollars and find the job responsible.
- Profile that job and name the root cause from the evidence.
- Apply the fix, then prove the saving two ways: the live task-time drop and the dollars on the after snapshot.

Everything you need was covered across the course: measure first (Part 02), the driver and UDF pitfalls (Part 08), and cost attribution (Part 10).

Each section provides the exact code to copy and paste if you get stuck.

> Serverless-first: this lab runs entirely on a serverless notebook. The lab data lives in your own `perf_lab` schema and does not touch the demo data.

## Learning Objectives

By the end of this lab, you will be able to:

1. **Query** a cost snapshot to rank workloads by DBU and dollars and find the most expensive job.
2. **Attribute** the cost to a workload using the identity and tag metadata on the usage record.
3. **Diagnose** the root cause by profiling the job and reading its Photon coverage.
4. **Apply** the fix and re-measure honestly with the result cache off and a steady-state median.
5. **Report** the saving as both a `total_task_duration_ms` drop and a dollar delta on the after snapshot.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **SQL and Python, and was tested using Serverless V5**. The whole lab runs on serverless. Other compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It builds the **invoice_lines** table in your own **perf_lab** schema (CloudField subscription billing, isolated from the demo data), reuses the Part 10 cost-snapshot generator to build a **before** and an **after** billing snapshot in **perf_lab**, and defines the ranking, savings, benchmark, and proof helpers you will use.

In [ ]:
%run ./Includes/Classroom-Setup-11

## B. The Scenario
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-scenario { background: #F9F7F4; border-radius: 10px; overflow: hidden; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.ds-scenario-band { background: var(--accent); color: #fff; padding: 14px 20px; text-align: center; font-size: 18pt; font-weight: 700; }
.ds-scenario-body { padding: 18px 20px; }
.ds-scenario-q { font-size: 16pt; font-style: italic; color: #0b2026; margin-bottom: 12px; text-align: center; }
.ds-scenario-text { font-size: 14pt; color: #0b2026; line-height: 1.6; }
.ds-scenario-text ul { margin: 12px 0; padding-left: 18px; }
.ds-scenario-text li { margin-bottom: 8px; }
</style>

<div class="ds-row">
  <div class="ds-scenario coral">
    <div class="ds-scenario-band">CloudField's monthly bill jumped</div>
    <div class="ds-scenario-body">
      <div class="ds-scenario-q">"Our Databricks bill went up last month and nobody knows why. Find out what happened and fix it."</div>
      <div class="ds-scenario-text">You run the data platform for CloudField, the subscription SaaS platform from Lab 1. The estate is a handful of jobs, a pipeline, a warehouse and some ad-hoc work.
        <ul>
          <li>The bill climbed over the last month with no new workloads and no more customers.</li>
          <li>You have a cost snapshot shaped like `system_vw.billing.usage` and `system_vw.billing.list_prices`, because real billing lands 1 to 2 hours late.</li>
          <li>One workload is quietly responsible for most of the increase. Nobody has profiled it since it shipped.</li>
        </ul>
        <strong>Your task: rank the estate by cost, find the job responsible, diagnose why, fix it and prove the saving.</strong>
      </div>
    </div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How to work this lab
- This lab uses the whole loop: measure, diagnose, fix, prove. Treat every section as a task, not a walkthrough.
- The tools are the ones you already used. Ranking cost is the Part 10 dollarize join. Diagnosing the job is the Part 08 Query Profile read. Proving the win is the Part 02 honest benchmark plus the DBU proxy.
- Why a snapshot: `system_vw.billing.usage` lands 1 to 2 hours after a workload runs, too long for a class. The setup shipped a **before** snapshot (the estate now) and an **after** snapshot (a month after the fix ships), both in **perf_lab**. The queries are the real ones. Only the landing time is faked.
- The dollar figures in the snapshot are illustrative synthetic values, not live Databricks list prices. The lesson is the ranking, the diagnosis, and the delta, none of which depend on the exact numbers.
- If you get stuck, each section's step cards include Copy buttons with the exact code to paste into the TODO cells.

</details>

## C. Find the Most Expensive Job
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.ds-callout code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
.info-callout { --accent:#1B5162; --tint:#F8F9FC; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">$</div>
  <div>
    <div class="ds-banner-title">Rank the estate by dollars</div>
    <div class="ds-banner-tagline">Join usage to prices, group by workload, and sort by cost. The workload at the top is where optimization effort pays back the most.</div>
  </div>
</div>

<div class="ds-callout info-callout">
  <div class="ds-eyebrow">Your task</div>
  <div class="ds-callout-title">Dollarize the before snapshot and read the ranking.</div>
  <ul>
    <li>Join <code>billing_usage_before</code> to <code>list_prices_snapshot</code> on <code>sku_name</code> and <code>cloud</code>, keeping the current price where <code>price_end_time IS NULL</code>.</li>
    <li>Cost is <code>SUM(usage_quantity * pricing.default)</code>. Group by the workload and sort highest first.</li>
    <li>Read the identity and team that own the top row. That is who you go talk to.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The one join that ranks the estate
- `system_vw.billing.usage` records consumption in DBUs, not dollars. `system_vw.billing.list_prices` holds the price per DBU per SKU per cloud. Join them on `sku_name` and `cloud`, filter `price_end_time IS NULL` to keep the currently effective price, and cost is `SUM(usage_quantity * pricing.default)`.
- Everything else in cost management is what you `GROUP BY`. Group by the workload (job name, warehouse name, or notebook path) to rank the estate. The setup exposes this as `RANK_JOBS_SQL` and runs it through `job_ranking("before")`.

### Attribute the top row before you touch it
- The usage row carries who ran it: `identity_metadata.run_as` is the user or service principal, and `custom_tags.team` is the team a serverless usage policy tagged it with.
- Read those on the top workload. Cost you cannot attribute is cost you cannot fix, because you do not know whose job to profile.
- Here the ranking should point at one job that dwarfs the rest. That is your target for the rest of the lab.

### Why the snapshot, not live billing
- Billing lands 1 to 2 hours after a run. The snapshot lets the whole class see the same ranking now. The query text is identical to what you would run against the real `system_vw.billing.usage`.

</details>

### C1. The ranking query

This is the dollarize join from Part 10, pointed at the **before** snapshot and grouped by workload. Against the real system tables the text is identical: only the two table names change to `system_vw.billing.usage` and `system_vw.billing.list_prices`.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  COALESCE(u.usage_metadata.job_name, u.usage_metadata.run_name,
           u.usage_metadata.notebook_path, u.billing_origin_product) AS workload,
  u.identity_metadata.run_as AS run_as,
  u.custom_tags.team AS team,
  ROUND(SUM(u.usage_quantity), 2) AS total_dbus,
  ROUND(SUM(u.usage_quantity * p.pricing.default), 2) AS est_cost_usd
FROM system_vw.billing.usage u
JOIN system_vw.billing.list_prices p
  ON u.sku_name = p.sku_name
 AND u.cloud = p.cloud
 AND p.price_end_time IS NULL
GROUP BY 1, 2, 3
ORDER BY est_cost_usd DESC
</pre>
</div>

### C2. Rank the estate and find the target

Run the ranking. Read the top row: the workload name, who runs it, and which team owns it. That workload is what you diagnose next.

In [ ]:
job_ranking("before")

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What the ranking tells you
- **The top workload.** One job should sit clearly above the rest on `est_cost_usd`. Note its name, its `run_as`, and its `team`. That is the job you profile in Section D.
- **DBUs and dollars track together here.** Every workload in this estate runs on serverless, so more DBUs means proportionally more dollars. The dollar column is just the DBU column times the price per DBU.
- **You are ranking by cost here.** A job can be fast per run and still top the bill if it runs often or does more work than it needs to. Speed matters, but the scoreboard in this section is spend.
- **You have not fixed anything yet.** Finding the target from the numbers is the skill in this section. The cause comes next.

</details>

## D. Diagnose the Root Cause

You found the expensive job. Now run it, open its Query Profile, and answer three diagnostic questions.

### D1. Read the job, then profile it

**Your task:**
1. Read the nightly job's SQL (see below). Notice how it builds the `settlement_tier` label.
2. Run it once so a Query Profile exists.
3. Open the profile: click **See performance** under the cell, or open **Query History** and choose **See query profile**. Find the percent of task time in Photon and the operator that holds most of the task time.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  settlement_tier_bad(region, amount) AS settlement_tier,
  COUNT(*) AS invoice_lines,
  ROUND(SUM(amount), 2) AS total_amount
FROM perf_lab.invoice_lines
GROUP BY 1
ORDER BY total_amount DESC
</pre>
</div>

The registered UDF `settlement_tier_bad` calls this Python function:

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
def settlement_tier_bad(region, amount):
    if region is None or amount is None:
        return None
    if amount >= 500:
        band = "A"
    elif amount >= 100:
        band = "B"
    else:
        band = "C"
    return region[:2].upper() + "-" + band
</pre>
</div>

In [ ]:
invoice_prep_bad_df = spark.sql(INVOICE_PREP_BAD_SQL)
_bad_rows = invoice_prep_bad_df.count()
print(f"Nightly job returned {_bad_rows:,} settlement-tier groups.")
print("Look for: percent of task time in Photon, and whether the UDF operator runs in Photon or not.")

### D2. Answer these from the Query Profile
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 260px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-label { font-size: 14pt; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.ds-card code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
</style>

<div class="ds-row">
  <div class="ds-card amber">
    <div class="ds-card-label">Question 1</div>
    <div class="ds-card-title">Where does the task time go?</div>
    <ul>
      <li>What does the job actually do? It rolls up invoice lines by a derived label.</li>
      <li>In the profile, which operator holds most of the aggregated task time?</li>
      <li>Is the scan the cost, or is it something the query does per row?</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-label">Question 2</div>
    <div class="ds-card-title">Is Photon actually running?</div>
    <ul>
      <li>Check the percent of task time in Photon in the profile.</li>
      <li>Does the expensive operator run in Photon or outside it?</li>
      <li>What in the query would make Photon hand an operator back to the JVM?</li>
    </ul>
  </div>
  <div class="ds-card red">
    <div class="ds-card-label">Question 3</div>
    <div class="ds-card-title">What forces the fallback?</div>
    <ul>
      <li>Read the job's SQL. What does it call to build the <code>settlement_tier</code> label?</li>
      <li>A registered Python UDF runs row at a time and is not a Photon operator.</li>
      <li>Could a built-in expression compute the same label and stay in Photon?</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Answers

**Question 1: Where does the task time go?**
- The job rolls up 20M invoice lines by a derived `settlement_tier` label. That label is computed by `settlement_tier_bad(region, amount)`, a registered Python UDF.
- In the profile, the UDF operator holds most of the aggregated task time. The scan itself is fast. The cost is per-row Python execution and serialization, not the read.
- In the operator attributes you can see `pythonUDF0` listed next to `region` and `amount`. That is Spark's internal name for the Python UDF expression, it confirms which columns are being serialized out to Python row by row.

**Question 2: Is Photon actually running?**
- The percent of task time in Photon is low. The UDF operator runs outside Photon.
- Photon cannot execute a pickled Python UDF. When it hits one, it hands that operator back to the Spark JVM, which serializes every row out to Python and back, one row at a time.

**Question 3: What forces the fallback?**
- `settlement_tier_bad(...)` is a registered Python UDF. On serverless env 5, a registered Python UDF is pickled (row-at-a-time) by default.
- The label is just the first two letters of the region upper-cased, plus an amount band (A/B/C). `UPPER`, `SUBSTRING`, and a `CASE` expression compute it natively, so a rewrite with built-ins stays in Photon end to end and the task time collapses.

**Why this is a cost problem, not a correctness problem:**
- The UDF returns the right answer every night. But row-at-a-time serialization burns far more aggregated task time than the same logic as a built-in. More task time means more DBUs per run, every single night. That is why this one job dominates the bill.
- This is the Part 08 UDF ladder seen from the cost side.

</details>

<div style="width: 100%; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:56px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/><path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/><path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/><path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/><rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/><path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/><path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/><path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">You diagnosed this from the profile, which is the point. Now compare your reasoning against <strong>Genie Code</strong>. Ask it why the job is slow and expensive, then judge whether its answer matches the Photon-coverage evidence you gathered. Click <strong>Copy</strong> and paste the prompt into Genie Code in the notebook or SQL editor.</div>
  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #EEEDE9; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-cost-query" style="flex: 1;">Our most expensive job groups invoice_lines by a settlement_tier label from a Python UDF. How do I rewrite it to stay in Photon?</span>
    <button onclick="var text=document.getElementById('genie-cost-query').innerText; var ta=document.createElement('textarea'); ta.value=text; ta.style.position='fixed'; ta.style.opacity='0'; document.body.appendChild(ta); ta.select(); document.execCommand('copy'); document.body.removeChild(ta); var b=this; b.innerText='Copied!'; setTimeout(function(){ b.innerText='Copy'; }, 2000);" style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 6px 12px; font-size: 14pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>
<summary>See response</summary>

![Genie Code response](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/lab/Lab 2 - Genie Response.png)

</details>
</div>

## E. Fix and Re-Measure
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-steps { display: flex; flex-direction: column; gap: 22px; }
.ds-step { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 20px; border-top: 6px solid var(--accent, #4299E0); }
.blue { --accent:#4299E0; --tint:rgba(66,153,224,0.10); }
.green { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.amber { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
.coral { --accent:#FF5F46; --tint:rgba(255,95,70,0.10); }
.ds-step-head { display: flex; align-items: center; gap: 12px; margin-bottom: 8px; }
.ds-step-num { width: 36px; height: 36px; border-radius: 50%; background: var(--accent); color: #fff; font-size: 18pt; font-weight: 800; line-height: 1; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.ds-step-title { font-size: 18pt; font-weight: 700; color: #0b2026; }
.ds-step-text { font-size: 14pt; color: #0b2026; line-height: 1.6; }
.ds-step-text ul { margin: 0; padding-left: 20px; }
.ds-step-text li { margin-bottom: 8px; }
.ds-step-text code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
.ds-step-summary { padding: 16px 18px; background: #FFF6F4; border: 2px solid #FF5F46; border-radius: 10px; }
.ds-step-summary-title { font-size: 16pt; font-weight: 800; color: #FF5F46; margin-bottom: 8px; }
.ds-step-summary-text { font-size: 15pt; line-height: 1.6; color: #0b2026; }
</style>

<div style="font-size:15pt;line-height:1.6;margin-bottom:18px;">Write the following in the <strong>next cell</strong>. The output stays identical; only the cost drops.</div>

<div class="ds-steps">
<div class="ds-step green">
  <div class="ds-step-head"><div class="ds-step-num">1</div><div class="ds-step-title">Run the built-in rewrite</div></div>
  <div class="ds-step-text">
    <ul><li>The built-in version is exposed as <code>INVOICE_PREP_GOOD_SQL</code>. Run it so its Query Profile exists.</li></ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">invoice_prep_good_df = spark.sql(INVOICE_PREP_GOOD_SQL)
print(invoice_prep_good_df.count(), "groups")</pre></div>
  </div>
</div>
<div class="ds-step amber">
  <div class="ds-step-head"><div class="ds-step-num">2</div><div class="ds-step-title">Confirm same result</div></div>
  <div class="ds-step-text">
    <ul><li>The answer must be identical to <code>invoice_prep_bad_df</code>: same tiers, same counts, same totals.</li></ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">invoice_prep_good_df.orderBy("settlement_tier").display()
invoice_prep_bad_df.orderBy("settlement_tier").display()</pre></div>
  </div>
</div>
<div class="ds-step coral">
  <div class="ds-step-head"><div class="ds-step-num">3</div><div class="ds-step-title">Benchmark both versions</div></div>
  <div class="ds-step-text">
    <ul><li>Run the honest recipe: result cache off, warm-up discarded, steady-state median.</li></ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">median_bad  = benchmark_query(INVOICE_PREP_BAD_SQL, "before (pickled UDF)")
median_good = benchmark_query(INVOICE_PREP_GOOD_SQL, "after (built-in)")</pre></div>
  </div>
</div>
<div class="ds-step-summary">
  <div class="ds-step-summary-title">What success looks like</div>
  <div class="ds-step-summary-text">The built-in version returns the same rows. Its Query Profile shows no UDF operator, and the percent of task time in Photon is back near 100%.</div>
</div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why this fixes the bill
- **`INVOICE_PREP_GOOD_SQL`** replaces the pickled Python UDF with `CONCAT(UPPER(SUBSTRING(region, 1, 2)), '-', CASE WHEN amount >= 500 THEN 'A' WHEN amount >= 100 THEN 'B' ELSE 'C' END)`. These are native Photon operators, so the entire query stays vectorized with zero serialization.
- **Why the answer must match.** The rewrite must produce identical output. If the tiers, counts, or totals differ by even one row, the rewrite is wrong. Comparing both DataFrames sorted by `settlement_tier` makes a mismatch immediately visible.
- **`benchmark_query`** disables the result cache, runs a warm-up pass (discarded), then takes the steady-state median. This gives an honest wall-clock comparison. The built-in version should be significantly faster because no row leaves the Photon engine.

</details>

In [ ]:
# TODO: run INVOICE_PREP_GOOD_SQL, confirm the same answer and benchmark both

In [ ]:
invoice_prep_good_df = spark.sql(INVOICE_PREP_GOOD_SQL)
print(invoice_prep_good_df.count(), "groups")

In [ ]:
invoice_prep_good_df.orderBy("settlement_tier").display()
invoice_prep_bad_df.orderBy("settlement_tier").display()

In [ ]:
median_bad  = benchmark_query(INVOICE_PREP_BAD_SQL, "before (pickled UDF)")

# Capture the boundary timestamp before the good-job runs.
# prove_savings() uses it to label before vs after in system_vw.query.history.
from datetime import datetime, timezone
split_time = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S')

median_good = benchmark_query(INVOICE_PREP_GOOD_SQL, "after (built-in)")

## E2. Prove the Saving
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-steps { display: flex; flex-direction: column; gap: 22px; }
.ds-step { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 18px 20px; border-top: 6px solid var(--accent, #4299E0); }
.blue { --accent:#4299E0; --tint:rgba(66,153,224,0.10); }
.green { --accent:#00A972; --tint:rgba(0,169,114,0.10); }
.ds-step-head { display: flex; align-items: center; gap: 12px; margin-bottom: 8px; }
.ds-step-num { width: 36px; height: 36px; border-radius: 50%; background: var(--accent); color: #fff; font-size: 18pt; font-weight: 800; line-height: 1; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
.ds-step-title { font-size: 18pt; font-weight: 700; color: #0b2026; }
.ds-step-text { font-size: 14pt; color: #0b2026; line-height: 1.6; }
.ds-step-text ul { margin: 0; padding-left: 20px; }
.ds-step-text li { margin-bottom: 8px; }
.ds-step-text code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
</style>

<div style="font-size:15pt;line-height:1.6;margin-bottom:18px;">Run the two cells below. They prove the saving two ways: <strong>live task time</strong> (drops immediately) and <strong>dollars</strong> (on the after snapshot).</div>

<div class="ds-steps">
<div class="ds-step green">
  <div class="ds-step-head"><div class="ds-step-num">1</div><div class="ds-step-title">Task-time drop</div></div>
  <div class="ds-step-text">
    <ul><li>Pulls <code>total_task_duration_ms</code>, <code>read_bytes</code>, and wall time for both runs from <code>system_vw.query.history</code>.</li></ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">prove_savings(split_time=split_time)</pre></div>
  </div>
</div>
<div class="ds-step blue">
  <div class="ds-step-head"><div class="ds-step-num">2</div><div class="ds-step-title">Dollar saving (before vs after snapshot)</div></div>
  <div class="ds-step-text">
    <ul><li>Compares per-workload cost on the before and after snapshots. The invoice-prep job should show the largest <code>saved_usd</code> and a high <code>pct_saved</code>.</li></ul>
    <div style="margin-top:10px;border-radius:6px;overflow:hidden;border:1px solid #DCE0E2;"><div style="display:flex;justify-content:flex-end;background:#EEEDE9;padding:6px 12px;"><button onclick="var pre=this.parentElement.nextElementSibling;var ta=document.createElement('textarea');ta.value=pre.innerText;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.select();document.execCommand('copy');document.body.removeChild(ta);var b=this;b.innerText='\u2713 Copied!';setTimeout(function(){b.innerText='Copy';},2000);" style="background:#FF5F46;color:#FFFFFF;border:none;border-radius:6px;padding:6px 14px;font-size:14pt;font-weight:700;cursor:pointer;white-space:nowrap;">Copy</button></div><pre style="margin:0;padding:14px;background:#F9F7F4;font-family:Menlo,Consolas,monospace;font-size:14pt;line-height:1.6;color:#0b2026;overflow-x:auto;white-space:pre-wrap;">snapshot_savings()</pre></div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Reading the proof
- **`prove_savings()`** reads `total_task_duration_ms` from `system_vw.query.history` for both runs. Fewer task-seconds for the same answer means fewer DBUs. The ingestion lag is 5 to 10 minutes, so if it returns no rows, wait and re-run.
- **`snapshot_savings()`** compares per-workload cost on the before and after billing snapshots and shows `saved_usd` and `pct_saved`. The invoice-prep job should show the largest saving because the pickled UDF was the single biggest cost driver.
- **Task time is the DBU proxy on serverless.** DBUs scale with aggregated task time, so fewer task-seconds means fewer DBUs per run. Multiply the per-run saving by a nightly job and it compounds into the monthly line you saw in the ranking.

</details>
</div>

In [ ]:
# TODO: prove the task-time drop

In [ ]:
if 'split_time' not in globals():
    print("split_time not set. Run the benchmark cell above first.")
else:
    prove_savings(split_time=split_time)

In [ ]:
# TODO: prove the dollar saving (before vs after snapshot)

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What the proof should show
- **`total_task_duration_ms` drops sharply for the built-in version.** This is the headline and the live proof. The pickled UDF forced a Photon fallback and per-row serialization, which burned aggregated task time. The built-in stays in Photon, so the same answer takes a fraction of the task-seconds.
- **Task time is the DBU proxy on serverless.** DBUs scale with aggregated task time, so fewer task-seconds means fewer DBUs per run. Multiply that per-run saving by a nightly job and it compounds into the monthly line you saw in the ranking.
- **The dollars confirm it on the after snapshot.** `snapshot_savings()` shows the invoice-prep job dropping the most in `saved_usd`. The after snapshot represents the estate a month later, giving the billing data time to land.
- **Wall clock is noisy and inflated by warm-up.** Prefer `total_task_duration_ms` and the dollar delta as your reliable signals.
- **The answer did not change.** The tiers, counts, and totals are identical between the bad and good versions. A cost fix that changes the result is a bug. Always confirm the output matches before you claim the saving.
- **Ingestion lag.** `system_vw.query.history` lags 5 to 10 minutes. If a row is missing from the proof, wait and re-run. The snapshot dollars are available immediately because they were pre-generated.

</details>

## Conclusion

In this lab, you ran the whole workflow end to end on a real cost regression:

1. **Found the cost**: ranked the estate by dollars with the usage-to-list_prices join and attributed the top workload to its job and owner.
2. **Diagnosed the cause**: profiled the nightly job and read the Photon fallback that a pickled Python UDF forced.
3. **Applied the fix**: rewrote the label with built-in expressions so the query stays in Photon, and confirmed the answer did not change.
4. **Proved the saving**: watched `total_task_duration_ms` drop live, then read the dollars saved on the after snapshot.

That is the same measure-diagnose-fix-prove loop the course teaches, applied to a real cost regression. You also saw that Genie Code could reach the same conclusion, and because you diagnosed it yourself first, you can judge whether its answer is right.

You have finished the content. From here:

- Take the **course assessment** to check your understanding against scenario-based questions.
- Keep the **Databricks performance and cost documentation** close as your reference: the performance efficiency best practices, the optimization recommendations, and the Predictive Optimization and cost-management pages.
- If class is out of time, this lab is yours to finish on your own. The step cards include Copy buttons, and the setup rebuilds everything on its own.

Going forward, measure cost with the same rigor you measure latency. Let the platform handle what it can (serverless, Predictive Optimization), and when a workload drifts, use this same loop to find it, diagnose it, fix it, and prove the delta.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>